In [ ]:
#| default_exp ship

In [ ]:
#| hide
from fastcore.test import *

A thin Mac app is kavacha's launcher, the pinned uv, a signed bootstrap lock and three generated files, in one bundle. This module puts them together, signs the helper and then the app, notarizes and staples it, zips it, uploads it to the release `app-v<V>` and writes the Homebrew cask. The launcher is a kavacha release (`Thin.launcher`, checked against `Thin.launcher_sha256`), or a local build of `swift/` for a test.

In [ ]:
#| export
from __future__ import annotations
import argparse, hashlib, json, os, re, shutil, subprocess, tarfile, tempfile, time, urllib.request
from fastcore.all import Path
from kavacha.thin import Thin
from kavacha.lock import Lock, sh, sha256

### The pieces

`fetch_uv` downloads the pinned uv and checks it against the pinned sha256 before it unpacks it.

In [ ]:
#| export
UV_URL = 'https://github.com/astral-sh/uv/releases/download/{v}/uv-aarch64-apple-darwin.tar.gz'
KAVACHA_DL = 'https://github.com/vedicreader/kavacha/releases/download'

def fetch_uv(lk, out=None):
    "The pinned uv in the app's build folder, downloaded and checked when it is not there. Its path."
    out = Path(out or lk.root/lk.t.build_dir/'uv'); out.mkdir(parents=True, exist_ok=True)
    v, want = lk.pinned_version(), (lk.root/lk.t.uv_sha256).read_text().split()[0]
    if (uv := out/'uv').is_file() and sh(uv, '--version').split()[1] == v: return uv
    with urllib.request.urlopen(UV_URL.format(v=v), timeout=120) as r: (tgz := out/'uv.tar.gz').write_bytes(r.read())
    if (got := sha256(tgz)) != want: raise SystemExit(f'the uv download is {got}, not the pinned {want}')
    with tarfile.open(tgz) as t:
        for m in t.getmembers():
            if m.isfile() and Path(m.name).name in ('uv', 'uvx'): m.name = Path(m.name).name; t.extract(m, out, filter='data')
    if sh(uv, '--version').split()[1] != v: raise SystemExit(f'{uv} is not uv {v}')
    return uv

def launcher(t, out, local=None):
    "The launcher binary: `local`, else the kavacha release `t.launcher`, checked against `t.launcher_sha256`."
    if local: return Path(local)
    if not (t.launcher and t.launcher_sha256): raise SystemExit('no launcher: give --launcher, or set launcher and launcher_sha256 in the spec')
    with urllib.request.urlopen(f'{KAVACHA_DL}/{t.launcher}/kavacha-launcher', timeout=120) as r: (f := Path(out)/'kavacha-launcher').write_bytes(r.read())
    if (got := sha256(f)) != t.launcher_sha256: raise SystemExit(f'the launcher is {got}, not the pinned {t.launcher_sha256}')
    f.chmod(0o755)
    return f

### The bundle

`bundle` lays out the app as `contract/api-1.md` names it. An app's `plist()` hook adds to the Info.plist, such as document types the app computes. It moves an old bundle aside, never into the trash of the person's Mac.

In [ ]:
#| export
def bundle(lk, out, exe, uv, lock, keys, icon=None):
    "The app bundle in `out`: the launcher, uv, the bootstrap lock, the keys, kavacha.json, childenv.json and Info.plist."
    t, out = lk.t, Path(out)
    app = out/f'{t.name}.app'
    if app.exists(): (trash := out/'trash').mkdir(parents=True, exist_ok=True); app.rename(trash/f'{app.name}-{os.getpid()}-{len(list(trash.iterdir()))}')
    c = app/'Contents'
    for d in ('MacOS', 'Helpers', 'Resources/Lock'): (c/d).mkdir(parents=True, exist_ok=True)
    shutil.copy(exe, c/'MacOS'/t.name); shutil.copy(uv, c/'Helpers'/'uv')
    gen = t.write(Path(tempfile.mkdtemp()), plist=(lk.hook('plist') or dict)())
    shutil.copy(gen/'Info.plist', c/'Info.plist')
    for f in ('kavacha.json', 'childenv.json'): shutil.copy(gen/f, c/'Resources'/f)
    (c/'Resources'/'childenv.json').write_text(json.dumps(lk.spec(), indent=2) + '\n')
    shutil.copy(keys, c/'Resources'/'lock-keys.pub')
    for f in ('manifest.json', 'manifest.sig', 'pyproject.toml', 'uv.lock', '.python-version'): shutil.copy(Path(lock)/f, c/'Resources'/'Lock'/f)
    icon = Path(icon or lk.root/t.icon) if (icon or t.icon) else None
    if icon and icon.is_file(): shutil.copy(icon, c/'Resources'/'AppIcon.icns')
    else: print(f'no icon at {icon}: the app has no icon')
    return app

`entitlements` writes the plist of the spec's entitlements; an app with none gets an empty one.

In [ ]:
#| export
def entitlements(t, out):
    import plistlib
    (f := Path(out)/'entitlements.plist').write_bytes(plistlib.dumps(dict(t.entitlements)))
    return f

In [ ]:
test_eq(Path(entitlements(Thin('Demo'), tempfile.mkdtemp())).read_bytes().count(b'<dict/>'), 1)

### Signing and shipping

`sign_app` signs the helper first, then the app: codesign signs nested code from the inside out. `-` signs ad hoc, for a local check only.

In [ ]:
#| export
def sign_app(app, identity='-', ents=None):
    "codesign the helper, then the app, with the hardened runtime."
    ts = '--timestamp' if identity != '-' else '--timestamp=none'
    sh('codesign', '--force', '--options', 'runtime', ts, '--sign', identity, Path(app)/'Contents'/'Helpers'/'uv')
    sh('codesign', '--force', '--options', 'runtime', ts, *(['--entitlements', ents] if ents else []), '--sign', identity, app)
    sh('codesign', '--verify', '--strict', '--deep', app)
    return app

def zip_app(app, z):
    Path(z).unlink(missing_ok=True)
    sh('ditto', '-c', '-k', '--sequesterRsrc', '--keepParent', app, z)
    return Path(z)

`cask` writes the Homebrew cask: the app's template with its `version` and `sha256` lines filled in, or a plain one from the spec.

In [ ]:
#| export
CASK = '''cask "{cask}" do
  version "{version}"
  sha256 "{sha}"

  url "https://github.com/{repo}/releases/download/app-v#{{version}}/{name}.zip"
  name "{name}"
  desc "{summary}"
  homepage "{homepage}"

  livecheck do
    url :url
    regex(/^app-v?(\\d+(?:\\.\\d+)+)$/i)
    strategy :github_releases
  end

  depends_on arch: :arm64
  app "{name}.app"
  uninstall quit: "{identifier}"
end
'''

def cask(lk, version, sha):
    "The cask text for app `version` whose zip has `sha`."
    t = lk.t
    f = t.cask_template and lk.root/t.cask_template
    if f and f.is_file():
        s = re.sub(r'^(\s*)version ".*"$', rf'\g<1>version "{version}"', f.read_text(), count=1, flags=re.M)
        return re.sub(r'^(\s*)sha256 ".*"$', rf'\g<1>sha256 "{sha}"', s, count=1, flags=re.M)
    return CASK.format(cask=t.cask, version=version, sha=sha, repo=t.repo, name=t.name, summary=t.summary, homepage=t.homepage or f'https://github.com/{t.repo}', identifier=t.identifier)

In [ ]:
c = cask(Lock(Thin('Demo', repo='me/demo', summary='A demo')), '1.0.0', 'ab')
test_eq(('version "1.0.0"' in c, 'sha256 "ab"' in c, 'uninstall quit: "org.demo.demo"' in c), (True, True, True))

### The flow

`build_mac` builds and signs a bundle and checks it with `verify_app`. `ship_mac` is the release: it refuses a dirty tree, a dev lock or a missing Developer ID; then it notarizes, staples, zips, uploads to `app-v<V>` with `--latest=false`, and writes the cask. Notary credentials are a keychain profile (`--profile`) or an App Store Connect key.

In [ ]:
#| export
def build_mac(lk, version=None, build=None, lock=None, keys=None, exe=None, identity='-', out=None, sign=True):
    "Build, sign and check the app; the app's path."
    t = lk.t
    if version: t.version = version
    if build: t.build = str(build)
    out = Path(out or lk.root/t.build_dir)
    uv = fetch_uv(lk)
    lock = Path(lock or out/'dev-lock')
    keys = Path(keys or (lock/'keys.pub' if (lock/'keys.pub').is_file() else lk.keys_pub))
    if not (lock/'manifest.sig').is_file(): raise SystemExit(f'no signed lock in {lock}; run kavacha lock dev-lock {lock}, or give --lock')
    app = bundle(lk, out, launcher(t, out, exe), uv, lock, keys)
    if sign: sign_app(app, identity, entitlements(t, out))
    lk.verify_app(app, zip_app(app, out/f'{t.name}.zip'))
    return app

def _notary(profile=None, key=None, key_id=None, issuer=None):
    if profile: return ['--keychain-profile', profile]
    if not (key and key_id and issuer): raise SystemExit('no notary credentials: give --profile, or --key, --key-id and --issuer')
    return ['--key', key, '--key-id', key_id, '--issuer', issuer]

def notarize(nz, notary):
    "Submit `nz`, wait for Apple's verdict, and stop with Apple's own log unless it is Accepted; the submission id. `notarytool --wait` can exit 0 on a rejection, so the status is read, not the exit code."
    p = subprocess.run(['xcrun', 'notarytool', 'submit', str(nz), *notary, '--wait', '--output-format', 'json'], text=True, capture_output=True)
    try: r = json.loads(p.stdout)
    except ValueError: raise SystemExit(f'notarytool gave no verdict: {(p.stderr or p.stdout).strip()[-2000:]}')
    if r.get('status') != 'Accepted':
        log = subprocess.run(['xcrun', 'notarytool', 'log', str(r.get('id', '')), *notary], text=True, capture_output=True).stdout
        raise SystemExit(f"Apple did not accept submission {r.get('id')}: {r.get('status')}\n{log.strip()[-2000:]}")
    print(f"notarized: submission {r['id']} is Accepted")
    return r['id']

def staple(app, tries=6, wait=20):
    "Staple Apple's ticket to `app` and check it. Apple's ticket service can lag the verdict by a minute, so a failure is tried again before it stops the release."
    for i in range(tries):
        try: sh('xcrun', 'stapler', 'staple', app); sh('xcrun', 'stapler', 'validate', app); return
        except SystemExit as e:
            if i == tries - 1: raise
            print(f'stapling failed ({i + 1} of {tries}); trying again in {wait} s: {str(e)[-300:]}'); time.sleep(wait)

def ship_mac(lk, version, identity, profile=None, key=None, key_id=None, issuer=None, exe=None, tap=None):
    "Release app `version`: build with the stable channel's lock, sign, notarize, staple, zip, upload, write the cask."
    t = lk.t
    if not re.fullmatch(r'\d+(\.\d+)+', version): raise SystemExit('the version must be one such as 1.0.0')
    if not lk.keys_pub.is_file(): raise SystemExit(f'no {lk.keys_pub}; run kavacha lock key and commit the public keys')
    if sh('git', '-C', lk.root, 'status', '--porcelain'): raise SystemExit('the git tree has changes that are not committed')
    if not identity.startswith('Developer ID Application'): raise SystemExit('a release is signed with a Developer ID Application identity')
    lk.gh('release', 'list', '--limit', '1')   # a broken gh login stops the release here, not after minutes of notarizing
    out = lk.root/t.build_dir
    shutil.rmtree(out/'lock', ignore_errors=True)
    lock = lk.fetch_channel(out/'lock')
    app = build_mac(lk, version, lock=lock, keys=lk.keys_pub, exe=exe, identity=identity, out=out)
    nz = zip_app(app, out/f'{t.name}.notary.zip')
    notarize(nz, _notary(profile, key, key_id, issuer))
    staple(app)
    z = zip_app(app, out/f'{t.name}.zip'); nz.unlink()
    tag = f'app-v{version}'
    if lk.gh('release', 'view', tag, check=False): lk.gh('release', 'upload', tag, z, '--clobber')
    else: lk.gh('release', 'create', tag, z, '--latest=false', '--title', f'{t.name} app {version}', '--notes', f'The {t.name} Mac app {version}. It installs its packages from the signed lock on first run.')
    text = cask(lk, version, sha256(z))
    if tap: (Path(tap)/'Casks'/f'{t.cask}.rb').write_text(text)
    else: (out/f'{t.cask}.rb').write_text(text)
    print(f'{t.name} app {version} is on {tag}; the cask is written')
    return app

### The command line

`kavacha ship mac` builds a local app (ad hoc signed, a dev lock). `--release V` is the release flow.

In [ ]:
#| export
def main(argv=None):
    "kavacha ship mac [--release V]"
    a = argparse.ArgumentParser(prog='kavacha ship', description='Build and ship a thin app.')
    a.add_argument('target', choices=['mac']); a.add_argument('--root', default='.')
    a.add_argument('--version'); a.add_argument('--build'); a.add_argument('--lock'); a.add_argument('--keys'); a.add_argument('--launcher')
    a.add_argument('--sign', default='-'); a.add_argument('--out'); a.add_argument('--no-sign', action='store_true'); a.add_argument('--uv-only', action='store_true')
    a.add_argument('--release'); a.add_argument('--profile'); a.add_argument('--key'); a.add_argument('--key-id'); a.add_argument('--issuer'); a.add_argument('--tap')
    o = a.parse_args(argv)
    lk = Lock(Thin.load(o.root))
    if o.uv_only: return print(fetch_uv(lk))
    if o.release: return ship_mac(lk, o.release, o.sign, o.profile, o.key, o.key_id, o.issuer, o.launcher, o.tap)
    print(build_mac(lk, o.version, o.build, o.lock, o.keys, o.launcher, o.sign, o.out, not o.no_sign))